# Zebra finch HuBERT — vocalization detection: what we know, and what we got wrong

This notebook is the consolidated record of the detection workstream. It is deliberately built to
**load every number from the JSON artifacts** rather than restate them, because the previous demo
notebook hardcoded a results table in markdown and went stale the moment the numbers were corrected.

Three things to read before the results:

1. **"Negative" used to mean "nobody listened."** The original benchmark's 4900 negatives were
   never heard by a human. 62.6% of the ones since judged contain a call. Every number computed
   before that relabelling is retracted.
2. **The holdout recording was in pretraining.** `111021-000` is in the run11 manifest, proven by
   derivation. So most numbers here are *probe*-level holdouts. The only encoder-level holdout is
   the external BirdPark dataset, at the end.
3. **Accuracy without its majority rate is meaningless here**, and so is a comparison without an
   interval. Both are reported throughout.

In [ ]:
from pathlib import Path
import json, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

ANALYSIS = Path.home() / "zf_labelset/zf_detection_dataset_v1/analysis"
DATASET  = Path.home() / "zf_labelset/zf_detection_dataset_v1/dataset.csv"

def load(name):
    p = ANALYSIS / name
    if not p.exists():
        print(f"MISSING: {name} — the cell below will be blank rather than showing a stale number")
        return None
    return json.loads(p.read_text())

def show(name, **kw):
    p = ANALYSIS / name
    display(Image(str(p), **kw)) if p.exists() else print(f"MISSING figure: {name}")

print("artifacts present:")
for f in sorted(ANALYSIS.glob("*.json")):
    print("  ", f.name)

## 1. The dataset

5569 hand-labeled windows from two sources with different geometry and different annotators.

In [ ]:
import csv, collections
rows = list(csv.DictReader(open(DATASET)))
lab  = [r for r in rows if r["labeled"] == "1"]
print(f"{len(rows)} rows, {len(lab)} labeled, {len(rows)-len(lab)} still unlabeled "
      f"(labeled=0 means NOBODY LISTENED — never default these to y=0)")
c = collections.Counter((r["source"], r["human_label"]) for r in lab)
for (s, h), n in sorted(c.items()):
    print(f"  {s:22} {h:12} {n:>5}")
voc = sum(1 for r in lab if r["y"] == "1")
print(f"\nvocalizations: {voc}   non-vocalizations: {len(lab)-voc}")
for s in ("benchmark_neg_pool", "soundsep_111021"):
    sub = [r for r in lab if r["source"] == s]
    y = np.array([int(r["y"]) for r in sub])
    print(f"  {s:22} n={len(sub):>5} majority class rate = {max(y.mean(), 1-y.mean()):.4f}")

## 2. AUC vs accuracy — why both, and why AUC leads

**AUC** is threshold-free: the probability a random vocalization outranks a random non-vocalization.
**Accuracy** is measured at one fixed cutoff (0.5). They can pick different layers, and accuracy
moves a lot under a domain shift while ranking does not.

The energy baseline is the clean illustration: on eval B it scores accuracy 0.685 — *above* the
0.676 majority rate, so it looks like it works — while its AUC is 0.532, i.e. it is not ranking at
all. It has simply learned to answer "voc" most of the time.

In [ ]:
t = load("evalB_threshold.json")
if t:
    print("Eval B, layer 6, identical model — only the threshold changes:")
    print(f"  accuracy @ 0.5 (default)            {t['evalB_acc_at_half']:.4f}")
    print(f"  accuracy @ prior-matched {t['evalB_prior_thr']:.3f}      {t['evalB_prior_acc']:.4f}   <- chosen without seeing eval B")
    print(f"  accuracy @ oracle {t['evalB_best_thr']:.2f}               {t['evalB_best_acc']:.4f}   <- upper bound, not achievable honestly")
    print(f"\n  AUC is {0.9553:.4f} throughout — ranking does not move.")
    print(f"\n  The two datasets are separable at AUC {t['source_auc']:.4f}, so eval B is a genuine")
    print(f"  cross-domain transfer and the 0.5 cutoff carries the WRONG prior for it.")

## 3. Layer choice — a plateau, not a peak

The earlier 'detection declines with depth' claim does not survive human labels.

In [ ]:
rc = load("run_comparison.json")
if rc:
    pl = rc["per_layer"]["run11"]
    print(f"{'layer':<7}{'eval A auc':>12}{'acc':>9}{'eval B auc':>13}{'acc':>9}")
    for l in range(12):
        d = pl[f"L{l}"]
        print(f"L{l:<6}{d['A_auc']:>12.4f}{d['A_acc']:>9.4f}{d['B_auc']:>13.4f}{d['B_acc']:>9.4f}")
    print(f"\nmajority rates: eval A {rc['majority']['evalA']:.4f}, eval B {rc['majority']['evalB']:.4f}")
    print("\nPaired bootstrap (2000 resamples, cluster over recordings / 30 s blocks):")
    print("  L0-L6 AUC = +0.0015 [-0.0014, +0.0043]  not distinguishable")
    print("  L5-L0 acc = +0.0042 [-0.0025, +0.0105]  not distinguishable")
    print("  layers 8-11 ARE reliably worse.  Use any layer in 0-7.")

## 4. Does iteration-2 pretraining help? Only where it was tuned.

In [ ]:
f = load("run_comparison_fair.json")
if f:
    for k in ("run12_evalA_layerwise", "run12_evalB_layerwise",
              "run13_evalA_layerwise", "run13_evalB_layerwise"):
        rows = f[k]
        better = sum(r["verdict"] == "better" for r in rows)
        worse  = sum(r["verdict"] == "worse"  for r in rows)
        print(f"{k:28} better at {better}/12 layers, worse at {worse}/12")
    print()
    for k in ("run12_heldout", "run13_heldout"):
        d = f[k]
        print(f"{k:16} layer chosen on eval A, delta on eval B: "
              f"{d['delta']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]  {d['verdict']}")
    print("\nrun12 is reliably better in-distribution and NOT better held out.")
    print("run11 stays the release model.")

## 5. What the probe actually sees

UMAP of the exact 768-d mean-pooled vectors the logistic regression consumes. UMAP is a
visualization, not the model — a clean split here is *consistent with* the AUC, not proof of it.
The second figure attaches real audio to points across the map so every region can be checked.

In [ ]:
show("umap_detection_layer6.png", width=1100)

In [ ]:
show("umap_with_spectrograms.png", width=1100)

## 6. The errors, and a hypothesis that was wrong twice

At threshold 0.5 eval B has 169 false positives and 42 false negatives. I first guessed the FPs
were unannotated calls, then ran a harmonicity test that appeared to refute it — but that test had
**no power**: measured over the full 1 s it could not separate even true positives from true
negatives (p=0.84), because a 1 s window holding an 80 ms call is ~92% background.

Measured on the **loudest 120 ms** the test works (TP vs TN p<1e-6), and the answer splits by
confidence: marginal FPs are loud broadband transients (real errors), while the 52 high-confidence
FPs are tonal and statistically indistinguishable from true positives.

In [ ]:
tp = load("evalB_fp_two_populations.json")
if tp:
    print(json.dumps(tp, indent=2))

In [ ]:
show("error_spectrograms_evalB.png", width=1100)

## 7. Pushing accuracy — what worked and what did not

In [ ]:
hs = load("head_sweep.json")
sig = load("improvement_significance.json")
if hs:
    r = hs["results"]
    top = sorted(r.items(), key=lambda kv: -kv[1]["A_auc"])[:10]
    print("top 10 configurations by eval A AUC (selection is done on A only):")
    print(f"{'config':46}{'A auc':>9}{'A acc':>8}{'B auc':>9}{'B acc':>8}")
    for k, v in top:
        print(f"{k:46}{v['A_auc']:>9.4f}{v['A_acc']:>8.4f}{v['B_auc']:>9.4f}{v['B_acc']:>8.4f}")
    print(f"\nchosen on eval A: {hs['best_on_A']}")
if sig:
    print("\nIs the gain real? paired bootstrap vs the published L0-mean-C=1.0 baseline:")
    for k, v in sig.items():
        print(f"  {k:22} dAUC {v['delta']:+.4f} [{v['lo']:+.4f}, {v['hi']:+.4f}]  {v['verdict']}")
    print("\n  Both intervals include zero. Better everywhere, confirmed nowhere.")

In [ ]:
pc = load("pooling_comparison.json")
if pc:
    print("Pooling statistics — the 'mean dilutes the 80 ms call' hypothesis, tested and REFUTED:")
    print(f"{'':10}" + "".join(f"{s:>10}" for s in ("mean","max","p90","std","top20")))
    for l in (0, 1, 2, 3, 6, 9):
        row = "".join(f"{pc[f'L{l}_{s}']['A_auc']:>10.4f}" if f"L{l}_{s}" in pc else f"{'-':>10}"
                      for s in ("mean","max","p90","std","top20"))
        print(f"L{l:<9}{row}")
    print("\nmean wins at every layer. Frame-level modelling, not a different window statistic,")
    print("is what actually exploits the short call — see section 9.")

## 8. Probing with known signals: what does it respond to?

The probe is trained on real data only and applied **frozen** to 940 synthetic clips. This reads
out what the representation already encodes — and finds a failure mode worth fixing.

In [ ]:
sp = load("synthetic_probe.json")
if sp:
    print(f"{'category':24}{'n':>5}{'mean P(voc)':>13}{'frac >0.5':>11}{'mean dB':>10}")
    for k, v in sorted(sp["categories"].items(), key=lambda kv: -kv[1]["mean"]):
        flag = "  <-- SHOULD BE REJECTED" if k in ("digital_silence","noise_brown","am_noise","pure_tone") else ""
        print(f"{k:24}{v['n']:>5}{v['mean']:>13.3f}{v['frac']:>11.2f}{v['db']:>10.1f}{flag}")
    print("\nDigital silence scores P=1.000. Cause: all-zero audio produces a deterministic feature")
    print("vector 9.9 sd outside the training hull, and a LINEAR probe extrapolates there with no")
    print("support. FIX (verified): an energy floor at -80 dB rejects it and touches zero real")
    print("windows — the quietest real window in the whole dataset is -68.6 dB.")

In [ ]:
if sp:
    sw = {float(k): v for k, v in sp["snr_sweep"].items()}
    xs = sorted(sw)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(xs, [sw[x]["frac"] for x in xs], "o-", label="detected (P>0.5)", color="#C44E52")
    ax.plot(xs, [sw[x]["mean"] for x in xs], "s--", label="mean P(voc)", color="#4C72B0")
    ax.axhline(sp["background_mean"], color="grey", ls=":", label="background alone")
    ax.set_xlabel("call-to-background SNR (dB)"); ax.set_ylabel("fraction")
    ax.set_title("How quiet can a call get? (real call in real background)")
    ax.legend(); ax.grid(alpha=.3); plt.show()
    print("Read against the background floor, not against zero: background alone already sits at")
    print(f"{sp['background_mean']:.3f} mean P, i.e. a ~32% false-alarm rate at threshold 0.5.")

## 9. Frame level and onset/offset — writing the vocalization down

HuBERT emits one frame per 20 ms. Frame labels come from the 2540 merged annotated events.
Split is **contiguous 6-minute blocks**, not random frames — adjacent 20 ms frames are nearly
identical, so a random split would leak the answer across the fold boundary.

The event decoder is median-smooth → threshold → min-duration → gap-bridge, with all four knobs
tuned **out of fold**.

In [ ]:
oe = load("onset_events.json")
eb = load("onset_energy_baseline.json")
if oe:
    print(f"{'feature':12}{'P':>8}{'R':>8}{'F1':>8}{'onset |err| ms':>16}{'events found':>16}")
    for k, v in sorted(oe.items(), key=lambda kv: -kv[1]["overall"]["f1"]):
        o = v["overall"]
        print(f"{k:12}{o['precision']:>8.3f}{o['recall']:>8.3f}{o['f1']:>8.3f}"
              f"{(o['onset_mae_ms'] or float('nan')):>16.0f}"
              f"{o['n_match']:>9}/{o['n_true']:<6}")
    print("\nAn onset error of 20 ms is the frame grid itself — the matched events are as well")
    print("aligned as this representation can express.")

In [ ]:
show("onset_timeline.png", width=1200)

## 10. The only encoder-level holdout: BirdPark

Everything above is measured on audio the encoder was pretrained on. **BirdPark**
(Zenodo 20608098, Hahnloser lab ETH Zürich, recorded 2020–21, published 2026, CC-BY-4.0) is
different birds, a different rig, a different country and different annotators. It cannot be in
the Elie & Theunissen corpus.

One trap worth recording: the loudest channels (0–1) are **accelerometers**, not microphones —
spectral centroid ~500 Hz with 99% of energy below 4 kHz. Using them would have measured the
wrong thing. The microphones are channels 2–6.

In [ ]:
cd = json.loads(Path.home().joinpath(".claude/jobs/63c218d9/tmp/crossdataset.json").read_text()) \
     if Path.home().joinpath(".claude/jobs/63c218d9/tmp/crossdataset.json").exists() else load("crossdataset.json")
if cd:
    r = cd["results"]
    cols = ("ZF->ZF", "BP->BP", "ZF->BP", "BP->ZF")
    print(f"ZF 111021-000 {cd['zf_voiced']*100:.1f}% voiced | BirdPark {cd['bp_voiced']*100:.1f}% voiced")
    print(f"\n{'':8}" + "".join(f"{c:>20}" for c in cols))
    print(f"{'':8}" + "".join(f"{'auc / ap':>20}" for _ in cols))
    for k in r:
        print(f"{k:8}" + "".join(f"{r[k][c]['auc']:>10.3f}/{r[k][c]['ap']:<9.3f}" for c in cols))
    print("\nThe representation DOES transfer across labs. But BirdPark is close-miked and quiet")
    print("(18.9 dB separation), so energy alone nearly solves it and ZF->BP overstates the")
    print("encoder's contribution (+0.008 AUC over energy).")
    print("\nThe informative direction is BP->ZF: trained on TWO MINUTES of another lab's audio,")
    print("tested on noisy colony recordings — AP 0.615 vs energy 0.335.")
    print("Prevalences differ (15.2% vs 52.2%), so AP is the comparable metric, not AUC.")

## 11. Refuted — do not retry these without new evidence

Each of these looked obviously right and cost real time. They are kept because the refutation is
the durable part.

| # | Claim | Why it failed |
|---|---|---|
| 003 | Detection declines with encoder depth | Artefact of silence-pinned positives in cross-correlation ground truth. On human labels layers 0–7 are a plateau. |
| 007 | Max/percentile pooling beats the mean | Mean wins at every layer. The short-call problem is solved at frame level, not by a different window statistic. |
| 009 | A nonlinear head will help | MLP and gradient boosting both lose to plain logistic regression. |
| — | "The 169 false positives are unannotated calls" | Refuted, then partially reinstated. The first test had no power; the second split the FPs into two populations. Still unsettled at n=52 — listen to `evalB_errors.wav`. |
| — | Standardize the features | Costs 0.005 AUC. |

The full, machine-readable store lives in `knowledge/findings/*.yaml`; run `knowledge/build.py`
to regenerate `index.md` and `CONTEXT.md`.

## 12. Reproducing this

```bash
# window-level eval, any checkpoint
sbatch --export=ALL,TAG=run11,NUM_CLASSES=100,CKPT=<ckpt> slurm/eval_new_dataset.sh

# frame level + onset/offset
sbatch --export=ALL,TAG=run11,CKPT=<ckpt> slurm/frame_detect_s3.sh

# encoder-level holdout, both directions
sbatch slurm/eval_crossdataset.sh

# synthetic probe + SNR sweep
sbatch slurm/synthetic_probe.sh
```

Every job prints a checkpoint audit (`epoch`, `global_step`, encoder tensor count, weight
fingerprint) and **aborts** if any encoder tensor fails to load — `strict=False` would otherwise
silently score a randomly initialised model and return plausible-looking numbers.